In [ ]:
import os

import itertools
import random
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn

from shapely.geometry import Point
import jismesh.utils as ju

import pyproj

In [ ]:
def parse_time(timestamp):
    hour = timestamp // 3600
    minute = (timestamp % 3600) // 60
    seconds = (timestamp % 3600) % 60
    return hour, minute, seconds

def Coordinate2MeshCodePandas(lon, lat, level):

    return ju.to_meshcode(lat, lon, level).astype(int).astype(str)

from math import radians, cos, sin, asin, sqrt, ceil, log


R = 6371


def haversine(lon1, lat1, lon2, lat2):
    # UNIT: KILOMETER

    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * asin(sqrt(a))

    return c * R

def mesh_distance(mesh_1, mesh_2):
    """

    :param mesh_1:
    :param mesh_2:
    :return:
    """
    lat1, lon1 = ju.to_meshpoint(mesh_1, 0.5, 0.5)
    lat2, lon2 = ju.to_meshpoint(mesh_2, 0.5, 0.5)

    return haversine(lon1, lat1, lon2, lat2)

def mesh_distance_df(row):
    return mesh_distance(row['origin_agg_key_cd'], row['destination_agg_key_cd'])

In [ ]:
def mesh_pair_distance(m1, m2):
    """
    Return distance for central point of m1 and m2
    Args:
        m1: mesh id of m1
        m2: mesh id of m2
    Returns:
        d: distance
    """
    m1_lat, m1_lon = ju.to_meshpoint(m1, lat_multiplier=0.5, lon_multiplier=0.5)
    m2_lat, m2_lon = ju.to_meshpoint(m2, lat_multiplier=0.5, lon_multiplier=0.5)
    geodesic = pyproj.Geod(ellps='WGS84')
    faz, baz, d = geodesic.inv(m1_lon, m1_lat, m2_lon, m2_lat)
    d /= 1000.0
    return d

In [ ]:
def calculate_mape(row):
    epsilon = 1e-8  
    seq1 = row[[f'vol_{i}_obs' for i in range(5, 24)]].values
    seq2 = row[[f'vol_{i}_ppf' for i in range(5, 24)]].values
    mape = np.mean(np.abs((seq1 - seq2) / (seq1 + epsilon))) * 100
    return mape

In [ ]:
# toyama = pd.read_csv('./data/external_od/toyama_od_daily.txt', encoding='shift-jis')
toyama = pd.read_csv('./data/external_od/toyama_od_daily.txt', encoding='shift-jis')
toyama = toyama.drop([len(toyama)-1], axis=0)
toyama = toyama.astype({'dt': 'str','mesh_cd': 'int64', 'od_class': 'int8', 'trip_class': 'int8', 'origin_agg_key_cd': 'int64', 'destination_agg_key_cd': 'int64', 'main_mode': 'int8' })

# shizuoka = pd.read_csv('./data/external_od/shizuoka_od_daily.txt', names=toyama.columns, encoding='shift-jis')
shizuoka = pd.read_csv('./data/external_od/shizuoka_od_daily.txt', names=toyama.columns, encoding='shift-jis')
shizuoka = shizuoka.drop([len(shizuoka)-1], axis=0)
shizuoka = shizuoka.astype({'dt': 'str','mesh_cd': 'int64', 'od_class': 'int8', 'trip_class': 'int8', 'origin_agg_key_cd': 'int64', 'destination_agg_key_cd': 'int64', 'main_mode': 'int8' })

In [ ]:
# zone_mesh_map = pd.read_csv('${PFLOW_DATA_DIR}/downloads/22.csv', encoding='shift-jis').set_index('基準メッシュ・コード')['都道府県市区町村コード'].to_dict()
zone_mesh_map = pd.read_csv('./data/mesh/22.csv', encoding='shift-jis').set_index('基準メッシュ・コード')['都道府県市区町村コード'].to_dict()
# zone_mesh_map
# cityname = pd.read_csv('${PFLOW_DATA_DIR}/downloads/22.csv', encoding='shift-jis').set_index('都道府県市区町村コード')['市区町村名'].to_dict()
cityname = pd.read_csv('./data/mesh/22.csv', encoding='shift-jis').set_index('都道府県市区町村コード')['市区町村名'].to_dict()
# cityname

## daily coverage

In [ ]:
for group in shizuoka.groupby("dt"):
    # group[1].to_csv('./data/external_od/shizuoka_'+group[0]+'_od_meshod_500m_daily.csv', index=False)
    group[1].to_csv('./data/external_od/shizuoka_'+group[0]+'_od_meshod_500m_daily.csv', index=False)

In [ ]:
shizuoka = shizuoka.replace('＊', 1)
shizuoka = shizuoka.astype({'total_people': 'float'})

In [ ]:
shizuoka_outer1 = shizuoka[(shizuoka["trip_class"]==3)&(shizuoka["od_class"]==3)]
shizuoka_outer1['destination_mesh_lv3'] = shizuoka_outer1['destination_agg_key_name'].astype(float) / 10
shizuoka_outer1['destination_mesh_lv3'] = shizuoka_outer1['destination_mesh_lv3'].astype(int)
shizuoka_outer1['origin_city'] = shizuoka_outer1['origin_agg_key_cd']
shizuoka_outer1['destination_city'] =shizuoka_outer1['destination_mesh_lv3'].map(zone_mesh_map)
shizuoka_outer1 = shizuoka_outer1.dropna(subset=['destination_city'])
shizuoka_outer1['destination_city'] = shizuoka_outer1['destination_city'].astype(int)
shizuoka_outer1

In [ ]:
shizuoka_outer2 = shizuoka[(shizuoka["trip_class"]==3)&(shizuoka["od_class"]==2)]
shizuoka_outer2['origin_mesh_lv3'] = shizuoka_outer2['origin_agg_key_name'].astype(float) / 10
shizuoka_outer2['origin_mesh_lv3'] = shizuoka_outer2['origin_mesh_lv3'].astype(int)
shizuoka_outer2['origin_city'] = shizuoka_outer2['origin_mesh_lv3'].map(zone_mesh_map)
shizuoka_outer2['destination_city'] = shizuoka_outer2['destination_agg_key_cd']
shizuoka_outer2 = shizuoka_outer2.dropna(subset=['origin_city'])
shizuoka_outer2['origin_city'] = shizuoka_outer2['origin_city'].astype(int)
shizuoka_outer2

In [ ]:
shizuoka_outer = pd.concat([shizuoka_outer1, shizuoka_outer2], axis=0)
shizuoka_outer

In [ ]:
shizuoka_outer_city_df = (
    shizuoka_outer.groupby(['dt', 'origin_city', 'destination_city'], as_index=False)
    .agg(
        # pref_name=('pref_name', 'first'),
        # city_name=('city_name', 'first'),
        # mesh_cd=('mesh_cd', 'first'),
        # od_class=('od_class', 'first'),
        # trip_class=('trip_class', 'first'),
        # origin_agg_key_cd=('origin_mesh_lv3', 'first'),
        # origin_agg_key_name=('origin_mesh_lv3', 'first'),
        # destination_agg_key_cd=('destination_mesh_lv3', 'first'),
        # destination_agg_key_name=('destination_mesh_lv3', 'first'),
        total_people=('total_people', 'sum')
    )
)

shizuoka_outer_city_df.reset_index(drop=True, inplace=True)
shizuoka_outer_city_df

In [ ]:
shizuoka_inner = shizuoka[(shizuoka["od_class"]==2)&(shizuoka["trip_class"]==2)]
shizuoka_inner

In [ ]:
shizuoka_inner['origin_mesh_lv3'] = shizuoka_inner['origin_agg_key_name'].astype(float) / 10
shizuoka_inner['origin_mesh_lv3'] = shizuoka_inner['origin_mesh_lv3'].astype(int)
shizuoka_inner['destination_mesh_lv3'] = shizuoka_inner['destination_agg_key_name'].astype(float) / 10
shizuoka_inner['destination_mesh_lv3'] = shizuoka_inner['destination_mesh_lv3'].astype(int)
shizuoka_inner

In [ ]:
shizuoka_inner['origin_city'] = shizuoka_inner['origin_mesh_lv3'].map(zone_mesh_map)
shizuoka_inner['destination_city'] = shizuoka_inner['destination_mesh_lv3'].map(zone_mesh_map)
shizuoka_inner = shizuoka_inner.dropna(subset=['origin_city'])
shizuoka_inner['origin_city'] = shizuoka_inner['origin_city'].astype(int)
shizuoka_inner = shizuoka_inner.dropna(subset=['destination_city'])
shizuoka_inner['destination_city'] = shizuoka_inner['destination_city'].astype(int)
shizuoka_inner

In [ ]:
shizuoka_lv3 = {}
idx = 0

for group in shizuoka_inner.groupby(['dt', 'origin_mesh_lv3', 'destination_mesh_lv3']):

    dt = group[1]['dt'].tolist()[0]
    pref_name = group[1]['pref_name'].tolist()[0]
    city_name = group[1]['city_name'].tolist()[0]
    mesh_cd = group[1]['mesh_cd'].tolist()[0]
    od_class = group[1]['od_class'].tolist()[0]
    trip_class = group[1]['trip_class'].tolist()[0]
    origin_agg_key_cd = group[1]['origin_mesh_lv3'].tolist()[0]
    origin_agg_key_name = group[1]['origin_mesh_lv3'].tolist()[0]
    destination_agg_key_cd = group[1]['destination_mesh_lv3'].tolist()[0]
    destination_agg_key_name = group[1]['destination_mesh_lv3'].tolist()[0]
    total_people = sum(group[1]['total_people'].tolist())
#     {'dt':dt, 'pref_name':pref_name, 'city_name':city_name, 'mesh_cd':mesh_cd, 'od_class': od_class, 'trip_class': trip_class,
#                                        'origin_agg_key_cd':origin_agg_key_cd, 'origin_agg_key_name':origin_agg_key_name, 'destination_agg_key_cd':destination_agg_key_cd,
#                                        'destination_agg_key_name': destination_agg_key_name, 'total_people':total_people})
    shizuoka_lv3[idx] = [dt, pref_name, city_name, mesh_cd, od_class, trip_class, origin_agg_key_cd, origin_agg_key_name, destination_agg_key_cd, destination_agg_key_name, total_people]
    idx += 1
    if idx % 10000 == 0:
        print(idx)

# shizuoka_lv3

In [ ]:
shizuoka_inner_lv3_df = (
    shizuoka_inner.groupby(['dt', 'origin_mesh_lv3', 'destination_mesh_lv3'], as_index=False)
    .agg(
        # pref_name=('pref_name', 'first'),
        # city_name=('city_name', 'first'),
        # mesh_cd=('mesh_cd', 'first'),
        # od_class=('od_class', 'first'),
        # trip_class=('trip_class', 'first'),
        # origin_agg_key_cd=('origin_mesh_lv3', 'first'),
        # origin_agg_key_name=('origin_mesh_lv3', 'first'),
        # destination_agg_key_cd=('destination_mesh_lv3', 'first'),
        # destination_agg_key_name=('destination_mesh_lv3', 'first'),
        total_people=('total_people', 'sum')
    )
)

shizuoka_inner_lv3_df.reset_index(drop=True, inplace=True)
shizuoka_inner_lv3_df

In [ ]:
shizuoka_inner_city_df = (
    shizuoka_inner.groupby(['dt', 'origin_city', 'destination_city'], as_index=False)
    .agg(
        # pref_name=('pref_name', 'first'),
        # city_name=('city_name', 'first'),
        # mesh_cd=('mesh_cd', 'first'),
        # od_class=('od_class', 'first'),
        # trip_class=('trip_class', 'first'),
        # origin_agg_key_cd=('origin_mesh_lv3', 'first'),
        # origin_agg_key_name=('origin_mesh_lv3', 'first'),
        # destination_agg_key_cd=('destination_mesh_lv3', 'first'),
        # destination_agg_key_name=('destination_mesh_lv3', 'first'),
        total_people=('total_people', 'sum')
    )
)

shizuoka_inner_city_df.reset_index(drop=True, inplace=True)
shizuoka_inner_city_df

In [ ]:
outer = shizuoka_outer_city_df.rename(columns={"total_people": "outer_people"})
inner = shizuoka_inner_city_df.rename(columns={"total_people": "inner_people"})
shizuoka_city_lv = pd.merge(outer, inner, on=["dt", "origin_city", "destination_city"], how="outer")
shizuoka_city_lv["total_people"] = shizuoka_city_lv["outer_people"].fillna(0) + shizuoka_city_lv["inner_people"].fillna(0)
shizuoka_city_lv = shizuoka_city_lv[["dt", "origin_city", "destination_city", "total_people"]]
shizuoka_city_lv

In [ ]:
shizuoka_lv3_df['total_people'].hist(bins=100)

In [ ]:
df = pd.DataFrame.from_dict(shizuoka_lv3, orient='index', columns=['dt', 'pref_name', 'city_name', 'mesh_cd', 'od_class', 'trip_class', 'origin_agg_key_cd', 'origin_agg_key_name', 'destination_agg_key_cd', 'destination_agg_key_name', 'total_people'])
df.to_csv('./data/external_od/shizuoka_OD_mesh_lv3.csv', index=False)

In [ ]:
date = ['20191013', '20191014', '20191015', '20191016', '20191017', '20191018', '20191019',
        '20201011', '20201012', '20201013', '20201014', '20201015', '20201016', '20201017']

In [ ]:
shizuoka = shizuoka.replace('＊', 5)
shizuoka = shizuoka.astype({'total_people': 'float'})

In [ ]:
mesh = gpd.read_file('${PFLOW_DATA_DIR}/shapefile/shizuoka500mesh/500m_mesh_2018_22.shp')
mesh[['MESH_ID', 'SHICODE', 'PTN_2015', 'geometry']]
# mesh.columns
mesh_shicode ={}
for idx, row in mesh.iterrows():
    if row['MESH_ID'] not in mesh_shicode:
        mesh_shicode[row['MESH_ID']] = row['SHICODE']

In [ ]:
code = pd.DataFrame.from_dict(mesh_shicode, orient='index').reset_index().rename(columns={'index': 'origin_agg_key_cd', 0: 'shicode'})

In [ ]:
shizuoka_day = shizuoka[shizuoka['dt']=='20191013']
shizuoka_day

In [ ]:
data = shizuoka_day[shizuoka_day['city_name'].isin(['静岡市 駿河区', '静岡市 葵区', '富士市', '浜松市 中区', '沼津市', '裾野市'])]
data = data[data['od_class']==2]
data = data[data['trip_class']==2]
data

In [ ]:
data['dist'] = data.apply(mesh_distance_df, axis=1)

In [ ]:
data

In [ ]:
mesh_pair_distance(523705271, 523705281	)

In [ ]:
data[data['total_people']>1].plot.scatter(x='dist', y='total_people',s=1, figsize=(12,8))

In [ ]:
data['dist'].hist()

In [ ]:
data['total_people'].max()

In [ ]:
shizuoka_day['city_name'].unique()

In [ ]:
len(data[data['city_name']== '富士市'])*0.8

In [ ]:
data[data['city_name']=='静岡市 葵区']['total_people'].hist(range=(400,14000),bins=400)

In [ ]:
data[data['city_name']=='富士市']['total_people'].hist(range=(400,14000),bins=400)

In [ ]:
data[data['city_name']=='静岡市 駿河区']['total_people'].hist(range=(400,14000),bins=400)

In [ ]:
data[data['city_name']=='沼津市']['total_people'].hist(range=(500,14000),bins=400)

In [ ]:
data[data['city_name']=='浜松市 中区']['total_people'].hist(range=(400,14000),bins=400)

In [ ]:
data[data['city_name']=='裾野市']['total_people'].hist(range=(100,14000),bins=400)

In [ ]:
shizuoka_day = pd.merge(shizuoka_day, code, how='left').rename(columns={'shicode': 'origin_shicode'})
# set(shizuoka_day['origin_shicode'].tolist())

In [ ]:
code = code.rename(columns={'origin_agg_key_cd': 'destination_agg_key_cd'})
shizuoka = pd.merge(shizuoka, code, on='destination_agg_key_cd').rename(columns={'shicode': 'destination_shicode'})

In [ ]:
shizuoka = shizuoka[['dt', 'pref_name', 'city_name', 'mesh_cd', 'od_class', 'trip_class',
       'origin_agg_key_cd', 'origin_shicode', 'destination_agg_key_cd',
       'destination_shicode', 'main_mode', 'total_people']]

In [ ]:
shizuoka_day[shizuoka_day['origin_shicode']==22101]['destination_shicode'].tolist()

In [ ]:
for group in shizuoka_day.groupby(['origin_shicode', 'destination_shicode']):
    print(group[0])
    #print(group[1])
    #print(sum(group[1]['total_people']))
    #break

In [ ]:
len(shizuoka_day.groupby(['origin_shicode', 'destination_shicode']))

## Classification

In [ ]:
data = od[['trip_class', 'origin_agg_key_cd', 'destination_agg_key_cd', 'main_mode', 'total_people']]
data = data[data['trip_class'].isin([1, 2])]
data = data.drop(['trip_class'], axis=1)
data = data.reset_index(drop=True)

In [ ]:
locs = list(set(data['destination_agg_key_cd'].tolist()+data['origin_agg_key_cd'].tolist()))

In [ ]:
len(data) / count

In [ ]:
for group in data.groupby('origin_agg_key_cd'):
    print(len(group[1]))

In [ ]:
zones = gpd.read_file('${PFLOW_DATA_DIR}/shapefile/ToyamaZone/N03-20210101_16_GML/N03-21_16_210101.shp')
zones = zones.drop(['N03_002'], axis=1).rename(columns={'N03_007': 'ZONE_ID'})

idx = 0
od_volume = {}

path = '${PFLOW_DATA_DIR}/pflow/pseudo/trip/Toyama/'

for f in os.listdir(path):
    if f.endswith('.zip'):
        print(f)
        pseudo_od_counter(os.path.join(path, f), zones, od_volume)

In [ ]:
od_volume

In [ ]:
zones = gpd.read_file('${PFLOW_DATA_DIR}/shapefile/ShizuokaZone/N03-20210101_22_GML/N03-21_22_210101.shp')
zones = zones.drop(['N03_002'], axis=1).rename(columns={'N03_007': 'ZONE_ID'})

idx = 0
od_volume = {}

path = '${PFLOW_DATA_DIR}/pflow/Shizuoka/p-csv/'

for d in os.listdir(path):
    for root, ds, fs in os.walk(path + d):
        print(root)
        for f in fs:
            if f.endswith('.csv'):
                od_counter(os.path.join(root, f), zones, od_volume)

In [ ]:
shizuoka = pd.DataFrame.from_dict(od_volume, orient='index').reset_index().rename(columns={'index':'od', 0:'volume'})
shizuoka['origin'], shizuoka['destination'] = shizuoka.od.str
shizuoka = shizuoka[['origin', 'destination', 'volume']]
shizuoka.to_csv('./data/external_od/Shizuoka_pt_zone_od.csv', index=False)

## Shizuoka Pseudo PFLOW

In [ ]:
# Not regenerate LV3 observation df
shizuoka_lv3_df = pd.read_csv('./data/external_od/shizuoka_OD_mesh_lv3.csv')
shizuoka_lv3_df

In [ ]:
shizuoka_lv3_df = shizuoka_lv3_df[shizuoka_lv3_df['dt']==20201013]
# shizuoka_lv3_df[(shizuoka_lv3_df['origin_agg_key_cd']==52370548)&(shizuoka_lv3_df['destination_agg_key_cd']==52370548)]

In [ ]:
# root = './data/PseudoPFLOW/ver2.0/trip/22/'
root = "C:/Data/PseudoPFLOW/trip/"
# root = "C:/log/one_step_t_50_beta_1.5/trip/"
# names = [baseline(beta=2.0), beta0.1, beta1.0, beta4.0, city_t_0.001, city_t_1000, facility_t_0.001, mesh_t_0.001, mesh_t_1000, one_step, one_step_1000, one_step_0.001,
#         one_step_t_20, one_step_t_30, one_step_t_50, one_step_t_50_beta_1.0, one_step_t_50_beta_1.5
#         one_step_nosoftmax_multi_pref, one_step_nosoftmax_log_attract, one_step_nosoftmax_log_attract, one_step_t_100_beta_1.5_distance(km), one_step_t_50_beta_1.5_distance(km) 
shizuoka = []


for dirname in os.listdir(root):
    if dirname not in ['22', '23']:
        continue
    for filename in os.listdir(root+dirname+'/'):
        print(filename)
        df = pd.read_csv(root+dirname+'/'+filename, names=['pid', 'time', 'origin_lon', 'origin_lat', 'destn_lon', 'destn_lat', 'transport', 'puropse', 'occupation'])
        df['origin_mesh'] = Coordinate2MeshCodePandas(df['origin_lon'], df['origin_lat'], 3)
        df['destn_mesh'] = Coordinate2MeshCodePandas(df['destn_lon'], df['destn_lat'], 3)
        shizuoka.append(df)


# for filename in os.listdir(root):
#     # print(filename)
#     df = pd.read_csv(root+filename, names=['pid', 'time', 'origin_lon', 'origin_lat', 'destn_lon', 'destn_lat', 'transport', 'puropse', 'occupation'])
#     df['origin_mesh'] = Coordinate2MeshCodePandas(df['origin_lon'], df['origin_lat'], 3)
#     df['destn_mesh'] = Coordinate2MeshCodePandas(df['destn_lon'], df['destn_lat'], 3)
#     shizuoka.append(df)
    
shizuoka = pd.concat(shizuoka, axis=0)
shizuoka

In [ ]:
len(shizuoka.pid.unique())

In [ ]:
shizuoka['transport'].hist()

In [ ]:
shizuoka_pseudo = dict()
idx = 0
for group in shizuoka.groupby(['origin_mesh', 'destn_mesh']):
    shizuoka_pseudo[idx] = [group[0][0], group[0][1], len(group[1])]
    idx += 1
shizuoka_pseudo = pd.DataFrame.from_dict(shizuoka_pseudo, orient='index')
shizuoka_pseudo.columns = ['origin', 'destination', 'volume']
shizuoka_pseudo

In [ ]:
shizuoka_pseudo['volume'].value_counts()

In [ ]:
shizuoka_pseudo['origin'] = shizuoka_pseudo['origin'].astype(int)
shizuoka_pseudo['destination'] = shizuoka_pseudo['destination'].astype(int)

In [ ]:
# shizuoka_sb = shizuoka_lv3_df[['origin_mesh_lv3', 'destination_mesh_lv3', 'total_people']]
# shizuoka_sb = shizuoka_sb.rename(columns={"origin_mesh_lv3": "origin", 'destination_mesh_lv3': 'destination'})
shizuoka_sb = shizuoka_lv3_df[['origin_agg_key_cd', 'destination_agg_key_cd', 'total_people']]
shizuoka_sb = shizuoka_sb.rename(columns={"origin_agg_key_cd": "origin", 'destination_agg_key_cd': 'destination'})
shizuoka_sb

In [ ]:
merged_df = pd.merge(shizuoka_sb, shizuoka_pseudo, on=['origin', 'destination'], how='inner')
merged_df

In [ ]:
mape = np.mean(np.abs((merged_df['total_people'] - merged_df['volume']) / (merged_df['total_people']))) * 100
mape

In [ ]:
merged_df['total_people'].sum() , merged_df['volume'].sum()  

In [ ]:
merged_df[merged_df['total_people']>15000]

In [ ]:
# 提取需要绘图的两列
x = merged_df['total_people']
y = merged_df['volume']

# 计算相关系数
correlation = np.corrcoef(x, y)[0, 1]  # 相关系数矩阵中的第一行第二列
r_square =  x.corr(y) ** 2

# 绘制散点图
plt.figure(figsize=(8, 6))
plt.xscale('log')
plt.yscale('log')
plt.scatter(x, y, color='blue', alpha=0.7, label=f'r square: {r_square:.2f}')
plt.title('MeshOD Volume Comparison (1km)', fontsize=16)
plt.xlabel('Softbank Mesh OD', fontsize=14)
plt.ylabel('Pseudo PFLOW v2', fontsize=14)

# 在图上标注相关系数
plt.text(
    x.min() + (x.max() - x.min()) * 0.05,  # X 坐标，稍微偏移避免重叠
    y.max() - (y.max() - y.min()) * 0.1,  # Y 坐标，靠近顶部
    f'R square: {r_square:.2f}',
    fontsize=12,
    color='red'
)

plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
merged_df[merged_df['total_people']>20000]

In [ ]:
merged_df[merged_df['volume']>3000]

In [ ]:
merged_df['origin_city'] = merged_df['origin'].map(zone_mesh_map)
merged_df['destination_city'] = merged_df['destination'].map(zone_mesh_map)
# merged_df2['origin_city'] = merged_df2['origin'].map(zone_mesh_map)
# merged_df2['destination_city'] = merged_df2['destination'].map(zone_mesh_map)
# merged_df2

In [ ]:
shizuoka_pseudo['origin_city'] = shizuoka_pseudo['origin'].map(zone_mesh_map)
shizuoka_pseudo['destination_city'] = shizuoka_pseudo['destination'].map(zone_mesh_map)
shizuoka_pseudo = shizuoka_pseudo.dropna()
shizuoka_pseudo['origin_cityname'] = shizuoka_pseudo['origin_city'].map(cityname)
shizuoka_pseudo['destination_cityname'] = shizuoka_pseudo['destination_city'].map(cityname)
shizuoka_pseudo

In [ ]:
shizuoka_pseudo_city = (
    shizuoka_pseudo.groupby(["origin_city", "destination_city"], as_index=False)  # 分组，并保留列作为 DataFrame
    .agg(
        {
         'volume': 'sum'}  # 汇总人口列
    ) 
)
shizuoka_pseudo_city

In [ ]:
shizuoka_sb_city = shizuoka_city_lv[shizuoka_city_lv['dt']=='20201013']
shizuoka_sb_city

In [ ]:
merged_new_city = pd.merge(shizuoka_sb_city, shizuoka_pseudo_city, on=['origin_city', 'destination_city'], how='inner')
merged_new_city

In [ ]:
data =  merged_new_city[merged_new_city['origin_city']!=merged_new_city['destination_city']]
data['ratio'] = data['total_people'] / data['volume']
data = data.sort_values(by='ratio', ascending=False)
data

In [ ]:
data['origin_cityname'] = data['origin_city'].map(cityname)
data['destination_cityname'] = data['destination_city'].map(cityname)
data[['origin_cityname', 'destination_cityname', 'total_people', 'volume']].sort_values(by=['total_people'], ascending=False)

In [ ]:
shizuoka_pseudo[(shizuoka_pseudo['origin_city']==22133)&(shizuoka_pseudo['destination_city']!=22133)]['volume'].sum()

In [ ]:
merged_df_city = (
    merged_df.groupby(["origin_city", "destination_city"], as_index=False)  # 分组，并保留列作为 DataFrame
    .agg(
        {'total_people': 'sum',
         'volume': 'sum'}  # 汇总人口列
    ) 
)
merged_df_city

In [ ]:
merged_df_city[merged_df_city['total_people'] > 400000]

In [ ]:
merged_new_city = merged_new_city[(merged_new_city['total_people'] < 800000)&(merged_new_city['volume'] > 50)&(merged_new_city['total_people'] > 50)]

In [ ]:
merged_new_city[(merged_new_city['total_people']<300000)&(merged_new_city['total_people']>100000)]

In [ ]:
from sklearn.metrics import r2_score
# 提取需要绘图的两列
x = merged_new_city['total_people']
y = merged_new_city['volume']

# 计算相关系数
correlation = np.corrcoef(x, y)[0, 1]  # 相关系数矩阵中的第一行第二列
r_square =  x.corr(y) ** 2
r2 = r2_score(x, y)
mape = np.mean(np.abs((x - y) / x)) * 100

# 绘制散点图
plt.figure(figsize=(8, 6))
# plt.xscale('log')
# plt.yscale('log')
plt.scatter(x, y, color='red', alpha=0.3, label=f'correlation: {correlation:.2f}')
plt.scatter(x, y, color='red', alpha=0.3, label=f'MAPE: {mape:.2f}%')
plt.title('City Level OD Volume Comparison ', fontsize=16)
plt.xlabel('Softbank OD', fontsize=14)
plt.ylabel('Pseudo PFLOW v2 OD', fontsize=14)

# # 在图上标注相关系数
# plt.text(
#     x.min() + (x.max() - x.min()) * 0.05,  # X 坐标，稍微偏移避免重叠
#     y.max() - (y.max() - y.min()) * 0.1,  # Y 坐标，靠近顶部
#     f'R square: {r2:.2f}',
#     fontsize=12,
#     color='red'
# )

plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
mape = np.mean(np.abs((data['total_people'] - data['volume']) / data['total_people'])) * 100
mape

In [ ]:
merged_df_city[(merged_df_city['total_people']>50000) & (merged_df_city['volume']<10000)]

In [ ]:
merged_df_city[(merged_df_city['total_people']>400000)]

In [ ]:
merged_df_city[(merged_df_city['origin_city']==22133)]

In [ ]:
merged_df_city = merged_df_city[(merged_df_city['total_people']<600000)]
merged_df_city

In [ ]:
df = pd.read_csv(r'C:\Data\PseudoPFLOW\trip\22\trip_22133.csv', names=['pid', 'time', 'origin_lon', 'origin_lat', 'destn_lon', 'destn_lat', 'transport', 'puropse', 'occupation'])
df

In [ ]:
len(df.pid.unique())